In [43]:
#manipulação de dados
import pandas as pd
import numpy as np

#bibliotecas do sklearn para preprocessamento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [44]:
df = pd.read_csv("../datasets/dados_tuberculose.csv", encoding='latin1', sep=',')
df.head(2)

C:\Users\olive\AppData\Local\Temp\ipykernel_21052\4077824653.py:1: DtypeWarning: Columns (20,21,36,38,44,46,58,60,66,76,96) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../datasets/dados_tuberculose.csv", encoding='latin1', sep=',')


,TP_NOT,ID_AGRAVO,DT_NOTIFIC,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,DT_DIAG,ANO_NASC,NU_IDADE_N,...,AGRAVDROGA,AGRAVTABAC,TEST_MOLEC,TEST_SENSI,ANT_RETRO,BAC_APOS_6,TRANSF,UF_TRANSF,MUN_TRANSF,IDADE_ANOS
0,2,A169,2016-03-30,2016,26,261160.0,1497.0,2013-06-17,1976.0,4037.0,...,9.0,9.0,5.0,0.0,0.0,0.0,0.0,NaN,NaN,37.0
1,2,A169,2017-04-07,2017,23,230190.0,1531.0,2013-09-11,1970.0,4042.0,...,0.0,0.0,5.0,7.0,0.0,0.0,0.0,NaN,NaN,42.0


In [45]:
df.shape

(772270, 98)

Filtro sudeste e terceira idade com dados acima de 2022

In [46]:
sudeste = [31, 32, 33, 35]
df = df.loc[df['SG_UF'].isin(sudeste)]


df = df.loc[(df.IDADE_ANOS >= 60)]
df = df.loc[df["NU_ANO"] >= 2022]

df.shape

(13999, 98)

In [47]:
df[['SITUA_ENCE']].value_counts()

SITUA_ENCE
1.0           8204
3.0           1518
4.0           1173
2.0            782
5.0            528
6.0            493
8.0            204
7.0            109
10.0            74
9.0              7
0.0              1
Name: count, dtype: int64

Filtrar para casos de tuberculose e definir obitos e cura

In [48]:
df = df.loc[df["SITUA_ENCE"].isin([1,3])]

dict_desfecho = {3: 1, 1: 0}
df['SITUA_ENCE'] = df['SITUA_ENCE'].replace(dict_desfecho)
df.rename(columns = {'SITUA_ENCE': "OBITO"}, inplace = True)
df.shape

(9722, 98)

In [49]:

lista_variaveis_categoricas = ['SG_UF', 'CS_SEXO', 'CS_RACA', 'CS_ESCOL_N',
                           'TRATSUP_AT', 'TRATAMENTO', 'AGRAVDROGA',
                           'POP_RUA', 'POP_LIBER', 'BACILOSC_E',
                           'AGRAVALCOO', 'FORMA', 'CS_GESTANT', 'AGRAVTABAC']
lista_variaveis_numericas = ['IDADE_ANOS']

variavel_desfecho = ['OBITO']

# Novo DataFrame com as variáveis independentes e alvo
df = df[lista_variaveis_categoricas + lista_variaveis_numericas + variavel_desfecho]

df[lista_variaveis_categoricas] = df[lista_variaveis_categoricas].astype('category')

# Alteração do tipo do dado para numérico
df["IDADE_ANOS"] = df["IDADE_ANOS"].astype(float)

#Verificação dos tipos dos dados
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9722 entries, 275888 to 772260
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   SG_UF       9722 non-null   category
 1   CS_SEXO     9722 non-null   category
 2   CS_RACA     8465 non-null   category
 3   CS_ESCOL_N  8871 non-null   category
 4   TRATSUP_AT  8954 non-null   category
 5   TRATAMENTO  9722 non-null   category
 6   AGRAVDROGA  9610 non-null   category
 7   POP_RUA     9610 non-null   category
 8   POP_LIBER   9615 non-null   category
 9   BACILOSC_E  9722 non-null   category
 10  AGRAVALCOO  9639 non-null   category
 11  FORMA       9722 non-null   category
 12  CS_GESTANT  9456 non-null   category
 13  AGRAVTABAC  9617 non-null   category
 14  IDADE_ANOS  9722 non-null   float64 
 15  OBITO       9722 non-null   float64 
dtypes: category(14), float64(2)
memory usage: 363.4 KB


In [50]:
df.columns = df.columns.str.lower()
df.columns

Index(['sg_uf', 'cs_sexo', 'cs_raca', 'cs_escol_n', 'tratsup_at', 'tratamento',
       'agravdroga', 'pop_rua', 'pop_liber', 'bacilosc_e', 'agravalcoo',
       'forma', 'cs_gestant', 'agravtabac', 'idade_anos', 'obito'],
      dtype='object')

**Separação entre treino e teste e definição de preditores e alvo**

In [51]:
X = df.drop(columns = "obito")
y = df.obito.copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify = y, random_state=42)

In [52]:
#Verificação da proporção de abandono na base de treino e teste
display(y_train.value_counts(normalize = True))
display(y_test.value_counts(normalize = True))

obito
0.0    0.843917
1.0    0.156083
Name: proportion, dtype: float64

obito
0.0    0.843686
1.0    0.156314
Name: proportion, dtype: float64

In [53]:
#Verificação de valores ausentes na base:
print("Quantidade de missings no treino: ", X_train.isna().sum().sum())
print("Quantidade de missings no teste: ", X_test.isna().sum().sum())

Quantidade de missings no treino:  2794
Quantidade de missings no teste:  867


In [54]:
#Lista de variáveis numéricas
lista_variaveis_numericas = ["idade_anos"]

#Lista de variáveis categóricas
lista_variaveis_categoricas = ['sg_uf', 'cs_sexo', 'cs_raca', 'cs_escol_n',
                               'tratsup_at', 'tratamento', 'agravdroga',
                               'pop_rua', 'pop_liber', 'bacilosc_e',
                               'agravalcoo', 'forma', 'cs_gestant','agravtabac']

preenchimento de `média` e `moda` com dados de treino para não ter vazamento de dados, poderia usar algum transformer/pipeline mas seguindo desenho do curso

In [55]:
#Obtenção das médias na base de treino
medias_treino = X_train[lista_variaveis_numericas].mean()

#Aplicação das médias na base de treino e de teste
X_train[lista_variaveis_numericas] = X_train[lista_variaveis_numericas].fillna(medias_treino)
X_test[lista_variaveis_numericas] = X_test[lista_variaveis_numericas].fillna(medias_treino)

In [56]:
X_train[lista_variaveis_categoricas].mode().T

,0
sg_uf,35
cs_sexo,M
cs_raca,1.0
cs_escol_n,3.0
tratsup_at,1.0
tratamento,1
agravdroga,2.0
pop_rua,2.0
pop_liber,2.0
bacilosc_e,1.0


In [57]:
#Obtenção da moda nas bases de treino
modas_treino = X_train[lista_variaveis_categoricas].mode().iloc[0]

#Aplicação da moda nas base de treino e de teste
X_train[lista_variaveis_categoricas] = X_train[lista_variaveis_categoricas].fillna(modas_treino)
X_test[lista_variaveis_categoricas] = X_test[lista_variaveis_categoricas].fillna(modas_treino)

In [58]:
#Verificação de valores ausentes na base:
print("Quantidade de missings no treino: ", X_train.isna().sum().sum())
print("Quantidade de missings no teste: ", X_test.isna().sum().sum())

Quantidade de missings no treino:  0
Quantidade de missings no teste:  0


**Padronização**

→ Existem dois tipos principais de escalonamento, e ambas podem ser importadas pelo módulo preprocessing do sklearn:

* Normalização (MinMaxScaler): Redimensiona os valores de uma variável para um intervalo fixo, geralmente entre 0 e 1.
* Padronização (StandardScaler): Transforma os dados para que tenham uma média de 0 e um desvio padrão de 1.

In [59]:
#Criação do Scaler
standard_scaler = StandardScaler()

#Treino do Scaler na base de treino
standard_scaler.fit(X_train[lista_variaveis_numericas])

#Aplicação do Scaler na base de treino e de teste
X_train[lista_variaveis_numericas] = standard_scaler.transform(X_train[lista_variaveis_numericas])
X_test[lista_variaveis_numericas] = standard_scaler.transform(X_test[lista_variaveis_numericas])

In [60]:
X_test[lista_variaveis_numericas].head(5)

,idade_anos
613555,0.722143
607291,0.314014
655537,0.314014
657287,0.041927
654489,0.450057


Lembrar sempre do cuidado com o vazamento de dados, como a padronização é feita a partir dos dados de treino a média e o desvio padrão podem não ser o valor esperado de 0 e 1 nos dados de teste

In [61]:
X_test.describe().round(2)

,idade_anos
count,2431.00
mean,0.01
std,0.98
min,-1.18
25%,-0.77
50%,-0.23
75%,0.59
max,3.72


**One-hot encoder**

→ Transformamos as variáveis categóricas em variáveis binárias (dummies) através do One Hot Encoding (também da biblioteca do sklearn). Essa técnica cria colunas para cada categoria, com valores 0 ou 1, permitindo que algoritmos interpretem as categorias corretamente.

→ Para categorias binárias, é possível eliminar uma das colunas para evitar redundância (drop = “if_binary”).

In [62]:
X_test[lista_variaveis_categoricas].head()

,sg_uf,cs_sexo,cs_raca,cs_escol_n,tratsup_at,tratamento,agravdroga,pop_rua,pop_liber,bacilosc_e,agravalcoo,forma,cs_gestant,agravtabac
613555,33,F,1.0,8.0,2.0,1,2.0,2.0,2.0,2.0,2.0,1.0,5.0,2.0
607291,33,M,9.0,2.0,1.0,1,2.0,2.0,2.0,1.0,2.0,1.0,6.0,2.0
655537,33,M,4.0,2.0,2.0,2,2.0,2.0,2.0,3.0,2.0,1.0,6.0,1.0
657287,33,F,2.0,3.0,1.0,1,2.0,2.0,2.0,3.0,2.0,2.0,6.0,2.0
654489,35,M,1.0,3.0,1.0,1,2.0,2.0,2.0,2.0,2.0,1.0,6.0,2.0


In [63]:
#cria o one hot encoder
one_hot_encoder = OneHotEncoder(drop = "if_binary", sparse_output = False, handle_unknown='ignore')

#treinar o one hot encoding na base de treino
one_hot_encoder.fit(X_train[lista_variaveis_categoricas])

#aplicar na base de treino e de teste
X_train_encoded = pd.DataFrame(one_hot_encoder.transform(X_train[lista_variaveis_categoricas]),
                               columns=one_hot_encoder.get_feature_names_out(),
                               index=X_train.index)

X_test_encoded = pd.DataFrame(one_hot_encoder.transform(X_test[lista_variaveis_categoricas]),
                              columns=one_hot_encoder.get_feature_names_out(),
                              index=X_test.index)

In [64]:
X_test_encoded.head()

,sg_uf_31,sg_uf_32,sg_uf_33,sg_uf_35,cs_sexo_F,cs_sexo_I,cs_sexo_M,cs_raca_1.0,cs_raca_2.0,cs_raca_3.0,...,cs_gestant_1.0,cs_gestant_2.0,cs_gestant_3.0,cs_gestant_4.0,cs_gestant_5.0,cs_gestant_6.0,cs_gestant_9.0,agravtabac_1.0,agravtabac_2.0,agravtabac_9.0
613555,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
607291,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
655537,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
657287,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
654489,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


**Unir dados tratados**

In [65]:
#Drop das features categóricas originais
X_train.drop(columns = lista_variaveis_categoricas, inplace = True)
X_test.drop(columns = lista_variaveis_categoricas, inplace = True)

#Junta as variáveis numéricas com as categóricas
X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

In [66]:
X_test.head()

,idade_anos,sg_uf_31,sg_uf_32,sg_uf_33,sg_uf_35,cs_sexo_F,cs_sexo_I,cs_sexo_M,cs_raca_1.0,cs_raca_2.0,...,cs_gestant_1.0,cs_gestant_2.0,cs_gestant_3.0,cs_gestant_4.0,cs_gestant_5.0,cs_gestant_6.0,cs_gestant_9.0,agravtabac_1.0,agravtabac_2.0,agravtabac_9.0
613555,0.722143,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
607291,0.314014,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
655537,0.314014,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
657287,0.041927,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
654489,0.450057,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [67]:
#Unificando base de Treino
df_train = pd.concat([X_train, y_train], axis = 1)

#Salvando base de Treino
path_to_save_train = "../datasets/dados_treino.csv"
df_train.to_csv(path_to_save_train, index = False)

In [68]:
#Unificando base de Teste
df_test= pd.concat([X_test, y_test], axis = 1)

#Salvando base de Teste
path_to_save_test = "../datasets/dados_teste.csv"
df_test.to_csv(path_to_save_test, index = False)